In [ ]:
# from google.colab import drive
# drive.mount("/content/drive") # Colab specific (not needed locally)

In [ ]:
DS_OG_PATH = "./datasets/extended_diac_transcripts"

In [ ]:
!mkdir -p ./datasets

In [ ]:
!cp -r $DS_OG_PATH ./datasets/data

In [ ]:
DS_PATH = ./datasets/data"

In [ ]:
import pandas as pd

In [ ]:
train_ids = pd.read_csv(f"{DS_PATH}/train_split.csv")['Participant_ID'].astype(int).tolist()
dev_ids = pd.read_csv(f"{DS_PATH}/dev_split.csv")['Participant_ID'].astype(int).tolist()
test_ids = pd.read_csv(f"{DS_PATH}/test_split.csv")['Participant_ID'].astype(int).tolist()

In [ ]:
all_participant_ids = train_ids + dev_ids + test_ids

In [ ]:
len(all_participant_ids)

In [ ]:
participant_texts = {}

In [ ]:
for pid in all_participant_ids:
    filename = f"{DS_PATH}/data/{pid}_P/{pid}_Transcript.csv"
    try:
        df_transcript = pd.read_csv(filename)
    except FileNotFoundError:
        print(f"Transcript file for participant {pid} not found.")
        continue

    utterances = (
    df_transcript["Text"]
    .dropna()
    .astype(str)
    .str.strip()
    )
    utterances = utterances[utterances != ""].tolist()
    merged_text = " ".join(utterances)
    participant_texts[pid] = merged_text

print(f"Participant 300 merged text (start): {participant_texts.get(300, '')[:100]}...")

In [ ]:
labels_df = pd.read_csv(f"{DS_PATH}/detailed_lables.csv")

In [ ]:
labels_df["Participant"] = labels_df["Participant"].astype(int)
label_map = dict(zip(labels_df["Participant"], labels_df["Depression_label"]))

label_map = {int(k): int(v) for k, v in label_map.items()}

In [ ]:
example_ids = [300, 308]
for pid in example_ids:
    if pid in label_map:
        print(f"Participant {pid} PHQ-8 binary label: {label_map[pid]}")

In [ ]:
def chunk_text(text, max_tokens=150):
    """Split text into chunks of up to max_tokens words (tokens)."""
    words = text.split()  # simple tokenization by whitespace
    chunks = []
    for i in range(0, len(words), max_tokens):
        chunk = " ".join(words[i:i + max_tokens])
        chunks.append(chunk)
    return chunks

In [ ]:
def chunk_text_sliding(text, max_tokens=150, stride=75):
    """
    Split text into overlapping chunks using a sliding window.
    Each chunk contains up to `max_tokens` words, with `stride` overlap.
    For example, max_tokens=150, stride=75 -> 50% overlap.
    """
    words = text.split()
    chunks = []
    start = 0
    while start < len(words):
        end = start + max_tokens
        chunk = " ".join(words[start:end])
        chunks.append(chunk)
        if end >= len(words):
            break
        start += stride
    return chunks


In [ ]:
sample_text = participant_texts.get(300, "")
chunks = chunk_text_sliding(sample_text, max_tokens=150, stride=75)
print(f"Created {len(chunks)} overlapping chunks.")
print("First chunk preview:", chunks[0][:100])
print("Second chunk preview:", chunks[1][:100])


In [ ]:
example_text = participant_texts.get(300, "")
example_chunks = chunk_text(example_text, max_tokens=150)
print(f"Participant 300 has {len(example_chunks)} chunks. First chunk sample:\n{example_chunks[0][:100]}...")

In [ ]:
import re
import unicodedata

# Common disfluencies in spoken transcripts (conservative list)
DEFAULT_FILLERS = [
    "um", "uh", "umm", "uhh", "er", "erm", "hmm", "mm", "mmm",
    "you know", "i mean", "like"
]

def preprocess_ediac_text(
    text: str,
    *,
    lowercase: bool = True,
    remove_bracketed: bool = True,
    remove_timestamps: bool = True,
    normalize_repeats: bool = True,
    normalize_unicode: bool = True,
    remove_fillers: bool = False,
    fillers=DEFAULT_FILLERS,
    keep_contractions: bool = True,
) -> str:
    """
    Conservative preprocessing for E-DAIC / clinical interview transcripts.
    Keeps semantics (negations, pronouns) while removing transcript/ASR noise.
    """

    if text is None:
        return ""

    if normalize_unicode:
        text = unicodedata.normalize("NFKC", text)

    if text.strip().lower() in {"nan", "none", "null"}:
        return ""

    # 3) Remove bracketed annotations: [laughter], (laughs), <cough>, {noise}, etc.
    if remove_bracketed:
        # square brackets, parentheses, angle, curly
        text = re.sub(r"\[.*?\]|\(.*?\)|<.*?>|\{.*?\}", " ", text)

    if remove_timestamps:
        # 00:12, 01:02:33, 0:34.56, 12:34.5
        text = re.sub(r"\b\d{1,2}:\d{2}(?::\d{2})?(?:\.\d+)?\b", " ", text)


    text = re.sub(r"(?im)^\s*(ellie|interviewer|participant|speaker)\s*:\s*", " ", text)


    text = text.replace("’", "'").replace("“", '"').replace("”", '"').replace("`", "'")

    if lowercase:
        text = text.lower()

    if normalize_repeats:
        # collapse long punctuation runs: "!!!" -> "!", "???" -> "?"
        text = re.sub(r"([!?.,])\1{1,}", r"\1", text)
        # collapse long hyphen/underscore runs
        text = re.sub(r"[-_]{2,}", " ", text)
        # collapse long letter elongations: "soooo" -> "soo" (conservative)
        text = re.sub(r"([a-z])\1{3,}", r"\1\1", text)

    # e.g., *laugh*, <unk>, [unk], (inaudible)
    text = re.sub(r"\b(inaudible|unintelligible|unknown|unk)\b", " ", text)

    if remove_fillers and fillers:
        # Handle multiword fillers like "you know", "i mean"
        # Use word boundaries; remove surrounding extra spaces after.
        for f in sorted(fillers, key=len, reverse=True):
            f_esc = re.escape(f)
            text = re.sub(rf"\b{f_esc}\b", " ", text)


    # If you *don't* want contractions, expand them elsewhere (not recommended here).
    if not keep_contractions:
        # remove apostrophes (turn don't -> dont), but this can harm meaning sometimes
        text = text.replace("'", "")


    # Keep: letters, numbers, whitespace, and . , ! ? ' "
    # text = re.sub(r"[^a-z0-9\s\.\,\!\?\’\'\"]+", " ", text)
    allowed = r"[^A-Za-z0-9\s\.\,\!\?\']+"   # includes uppercase
    text = re.sub(allowed, " ", text)

    # 13) Collapse whitespace
    text = re.sub(r"\s+", " ", text).strip()


    return text


In [ ]:
train_data = []
dev_data = []
test_data = []

for pid in train_ids:
    text = participant_texts.get(pid, "")
    if text == "":
        continue
    label = label_map.get(pid, 0)
    cleaned = preprocess_ediac_text(text, remove_fillers=False)

    chunks = chunk_text_sliding(cleaned, max_tokens=150, stride=75)
    for idx, chunk in enumerate(chunks):
        train_data.append({
  "pid": pid,
  "chunk_id": idx,
  "text": chunk,
  "label": int(label)
})

for pid in dev_ids:
    text = participant_texts.get(pid, "")
    if text == "":
        continue
    label = label_map.get(pid, 0)
    cleaned = preprocess_ediac_text(text, remove_fillers=False)

    # chunks = chunk_text(text, max_tokens=150)
    chunks = chunk_text_sliding(cleaned, max_tokens=150, stride=75)
    for idx, chunk in enumerate(chunks):
        dev_data.append({
  "pid": pid,
  "chunk_id": idx,
  "text": chunk,
  "label": int(label)
})
for pid in test_ids:
    text = participant_texts.get(pid, "")
    if text == "":
        continue
    label = label_map.get(pid, 0)
    cleaned = preprocess_ediac_text(text, remove_fillers=False)

    # chunks = chunk_text(text, max_tokens=150)
    chunks = chunk_text_sliding(cleaned, max_tokens=150, stride=75)
    for idx, chunk in enumerate(chunks):
        test_data.append({
  "pid": pid,
  "chunk_id": idx,
  "text": chunk,
  "label": int(label)
})


print(f"Total training examples (chunks): {len(train_data)}")
print(f"Total dev examples (chunks): {len(dev_data)}")
print(f"Total test examples (chunks): {len(test_data)}")


In [ ]:
print(f"First training example:\n{train_data[0]}")

In [ ]:
missing = [pid for pid in all_participant_ids if pid not in label_map]
print("Missing labels:", len(missing), missing[:10])


In [ ]:
empty = [pid for pid in all_participant_ids if not participant_texts.get(pid, "").strip()]
print("Empty transcripts:", len(empty), empty[:10])


In [ ]:
import numpy as np
y = np.array([ex["label"] for ex in train_data])
print("Train depressed %:", (y==1).mean())


In [ ]:
patient_labels = [label_map[pid] for pid in train_ids]
print("Patient-level depressed %:", np.mean(patient_labels))


In [ ]:
import json

with open('train.jsonl', 'w', encoding='utf-8') as f:
    for example in train_data:
        json_line = json.dumps(example)
        f.write(json_line + "\n")

with open('dev.jsonl', 'w', encoding='utf-8') as f:
    for example in dev_data:
        f.write(json.dumps(example) + "\n")

with open('test.jsonl', 'w', encoding='utf-8') as f:
    for example in test_data:
        f.write(json.dumps(example) + "\n")

with open('train.jsonl', 'r') as f:
    for _ in range(3):
        line = f.readline().strip()
        print(line)


In [ ]:
import os
import shutil

output_dir_drive = "./data"

os.makedirs(output_dir_drive, exist_ok=True)

jsonl_files = ['train.jsonl', 'dev.jsonl', 'test.jsonl', 'train_cased.jsonl', 'dev_cased.jsonl', 'test_cased.jsonl', ]

for file_name in jsonl_files:
    source_path = file_name
    destination_path = os.path.join(output_dir_drive, file_name)
    try:
        shutil.copy(source_path, destination_path)
        print(f"Successfully copied {file_name} to {destination_path}")
    except FileNotFoundError:
        print(f"Error: {file_name} not found in the current directory.")
    except Exception as e:
        print(f"Error copying {file_name}: {e}")
